In [0]:
%pip install -U transformers torch torchvision mlflow accelerate vllm==0.7.0

In [0]:
dbutils.library.restartPython()

In [0]:
import mlflow
mlflow.set_registry_uri("databricks-uc")

In [0]:
import torch
import transformers
import pandas as pd
from transformers import AutoModel, AutoModelForCausalLM, AutoTokenizer, AutoConfig
from vllm import LLM, SamplingParams
import json
import os
import yaml

In [0]:
model_path = 'deepseek-ai/DeepSeek-R1-Distill-Llama-8B'
# model_path = "deepseek-ai/DeepSeek-R1-Distill-Qwen-1.5B"
model_cache_path = f"/local_disk0/models_cache/{model_path}"

In [0]:
from huggingface_hub import snapshot_download

snapshot_location = snapshot_download(repo_id=model_path, 
                                      local_dir=model_cache_path)
snapshot_location

### Try with deploying vLLM with mlflow


In [0]:
from mlflow.types.llm import CHAT_MODEL_INPUT_SCHEMA, CHAT_MODEL_OUTPUT_SCHEMA
from mlflow.models.signature import infer_signature, ModelSignature

signature = ModelSignature(inputs=CHAT_MODEL_INPUT_SCHEMA, outputs=CHAT_MODEL_OUTPUT_SCHEMA)
print(signature)

In [0]:
file_path = "deepseek_vllm_config.yaml"

config = {
  "tensor_parallel" : 1, # distilled qwen1.5 and llama8 need 1, may be qwen14b and 32b need to be 4 yet to test these 
  "dtype": "bfloat16", # using auto as of now, try to override in case customer only has T4 instances
  "max_model_len": 2048 # one more knob to control vllm models memory, this limits models context window.
}

with open(file_path, 'w') as file:
    yaml.dump(config, file)

In [0]:
tools = [{
    "type": "function",
    "function": {
        "name": "get_current_weather",
        "description": "Get the current weather in a given location",
        "parameters": {
            "type": "object",
            "properties": {
                "city": {
                    "type":
                    "string",
                    "description":
                    "The city to find the weather for, e.g. 'San Francisco'"
                },
                "state": {
                    "type":
                    "string",
                    "description":
                    "the two-letter abbreviation for the state that the city is"
                    " in, e.g. 'CA' which would mean 'California'"
                },
                "unit": {
                    "type": "string",
                    "description": "The unit to fetch the temperature in",
                    "enum": ["celsius", "fahrenheit"]
                }
            },
            "required": ["city", "state", "unit"]
        }
    }
}]

input_example = {
    "messages": [
        {"role": "user", "content": "What is the weather in San Francisco, CA in celsius?"},
    ],
    "tools": tools,
    "temperature": 0.1,
    "max_tokens": 100,
    "top_p": 0.9,
}

In [0]:
ds_model_path = os.path.join(os.getcwd(), "deepseep_mlflow_model.py")
config_path = os.path.join(os.getcwd(), file_path)

with mlflow.start_run():
    model_info = mlflow.pyfunc.log_model(
        "model",
        python_model=ds_model_path,
        artifacts={"model_path": model_cache_path},
        model_config=config_path,
        input_example=input_example,
        signature=signature,
        pip_requirements=["transformers","torch","torchvision","accelerate","mlflow==2.20.0", "vllm==0.7.0"]
    )

In [0]:
model_info.model_uri

### Test the model loading 

In [0]:
dbutils.library.restartPython()

In [0]:
import mlflow
mlflow.set_registry_uri("databricks-uc")

In [0]:
# model_version=1
# model_uri = f"models:/uc_sriharsha_jana.test_db.deepseek_qwen_v1_5b/{model_version}"
model_uri = 'runs:/35944af107f74e78b5f02191a1173975/model'
loaded_model = mlflow.pyfunc.load_model(model_uri)

In [0]:
tools = [
    {
        "type": "function",
        "function": {
            "name": "get_weather",
            "description": "Get weather of an location, the user shoud supply a location first",
            "parameters": {
                "type": "object",
                "properties": {
                    "location": {
                        "type": "string",
                        "description": "The city and state, e.g. San Francisco, CA",
                    }
                },
                "required": ["location"]
            },
        }
    },
]

In [0]:
input_example = {
    "messages": [
        {"role": "user", "content": "What is the weather in Bangalore, India? tell the answer in celcius"},
    ],
    "tools": tools,
    "temperature": 0.5,
    "max_tokens": 256,
    "top_p": 0.95,
}

response = loaded_model.predict(input_example)
response

### Register the model to UC

In [0]:
reg_model_info = mlflow.register_model(model_uri, "uc_sriharsha_jana.test_db.deepseek_vllm_llama_3_8b")
reg_model_info